# Lab 9: TextBlob Analysis

This notebook performs text analysis on product reviews using TextBlob.

In [ ]:
# Setup
import nltk
from textblob import TextBlob, Word
from nltk.corpus import stopwords
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import collections
import re
import numpy as np

# Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('wordnet', quiet=True)

# Sample product reviews dataset (text, label)
sample_reviews = [
    ('This product is amazing and works perfectly!', 'positive'),
    ('Terrible quality, broke after one use.', 'negative'),
    ('Good value for money, satisfied customer.', 'positive'),
    ('Very disappointed with the service.', 'negative'),
    ('Excellent build quality and fast delivery.', 'positive'),
    ('Not as described, poor customer support.', 'negative'),
    ('Highly recommend to everyone!', 'positive'),
    ('Waste of money, do not buy.', 'negative'),
    ('Perfect fit and comfortable.', 'positive'),
    ('Slow shipping and damaged item.', 'negative'),
    ('Best purchase ever, love it!', 'positive'),
    ('Okay product but overpriced.', 'neutral'),
    ('Super fast and reliable.', 'positive'),
    ('Expected better performance.', 'negative'),
    ('Great features for the price.', 'positive'),
    ('Functionality is subpar.', 'negative'),
    ('Outstanding product quality.', 'positive'),
    ('Average at best.', 'neutral'),
    ('Fantastic experience overall.', 'positive'),
    ('Major defects found.', 'negative')
]

print(f'Loaded {len(sample_reviews)} reviews')

## Section 1 – Noun Phrase Extraction

In [ ]:
# Extract noun phrases
noun_phrases = []
for review_text, label in sample_reviews:
    blob = TextBlob(review_text)
    noun_phrases.extend(blob.noun_phrases)

# Count frequency
top_nps = collections.Counter(noun_phrases).most_common(10)
nps, freqs = zip(*top_nps)

# Plot
plt.figure(figsize=(10,6))
plt.barh(nps, freqs, color='coral')
plt.xlabel('Frequency')
plt.ylabel('Noun Phrase')
plt.title('Top 10 Noun Phrases in Reviews')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Section 2 – Identifying Verbs and Nouns (POS Tagging)

In [ ]:
# POS categories
noun_tags = {'NN', 'NNS', 'NNP', 'NNPS'}
verb_tags = {'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ'}
adj_tags = {'JJ', 'JJR', 'JJS'}

nouns = []
verbs = []
adjs = []

for review_text, label in sample_reviews:
    blob = TextBlob(review_text)
    for word, pos in blob.tags:
        if pos in noun_tags:
            nouns.append(word)
        elif pos in verb_tags:
            verbs.append(word)
        elif pos in adj_tags:
            adjs.append(word)

# Top 10
noun_counter = collections.Counter(nouns).most_common(10)
verb_counter = collections.Counter(verbs).most_common(10)
adj_counter = collections.Counter(adjs).most_common(10)

# Plot
fig, axs = plt.subplots(1, 3, figsize=(18,6))

axs[0].barh(*zip(*noun_counter), color='blue')
axs[0].set_title('Top 10 Nouns')
axs[0].set_xlabel('Frequency')

axs[1].barh(*zip(*verb_counter), color='green')
axs[1].set_title('Top 10 Verbs')
axs[1].set_xlabel('Frequency')

axs[2].barh(*zip(*adj_counter), color='orange')
axs[2].set_title('Top 10 Adjectives')
axs[2].set_xlabel('Frequency')

plt.tight_layout()
plt.show()

## Section 3 – Tokenization

In [ ]:
# Tokenization stats
word_counts = []
sent_counts = []
all_words = []

for review_text, label in sample_reviews:
    blob = TextBlob(review_text)
    words = len(blob.words)
    sents = len(blob.sentences)
    word_counts.append(words)
    sent_counts.append(sents)
    all_words.extend(blob.words)

print(f'Total words: {len(all_words)}')
print(f'Unique words: {len(set(all_words))}')
print(f'Avg words per review: {np.mean(word_counts):.1f}')
print(f'Avg sentences per review: {np.mean(sent_counts):.1f}')

# Plot word count per review
plt.figure(figsize=(12,6))
plt.bar(range(len(word_counts)), word_counts, color='lightblue', edgecolor='black')
plt.xlabel('Review #')
plt.ylabel('Word Count')
plt.title('Word Count per Review')
plt.show()

## Section 4 – Lemmatization

In [ ]:
# Lemmatization
stop_words = set(stopwords.words('english'))
lemmatized_tokens = []

for review_text, label in sample_reviews:
    # Clean
    clean_text = re.sub(r'[^a-zA-Z\s]', '', review_text.lower())
    blob = TextBlob(clean_text)
    tokens = []
    for word_pos in blob.tags:
        word = word_pos[0].lower()
        if word not in stop_words:
            if word_pos[1] in verb_tags:
                tokens.append(Word(word).lemmatize('v'))
            else:
                tokens.append(Word(word).lemmatize())
    lemmatized_tokens.extend(tokens)

# Word cloud
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(' '.join(lemmatized_tokens))

plt.figure(figsize=(12, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of Lemmatized Reviews')
plt.show()

## Section 5 – Sentiment Analysis

In [ ]:
# Sentiment prediction
actual_sent = [label for text, label in sample_reviews]
pred_sent = []

for review_text, label in sample_reviews:
    blob = TextBlob(review_text)
    pol = blob.sentiment.polarity
    if pol > 0:
        pred_sent.append('positive')
    elif pol < 0:
        pred_sent.append('negative')
    else:
        pred_sent.append('neutral')

# Accuracy
accuracy = sum(p == a for p, a in zip(pred_sent, actual_sent)) / len(sample_reviews) * 100
print(f'Sentiment Accuracy: {accuracy:.1f}%')

# Plot distribution
sentiments = ['positive', 'negative', 'neutral']
actual_counts = [actual_sent.count(s) for s in sentiments]
pred_counts = [pred_sent.count(s) for s in sentiments]

x = np.arange(len(sentiments))
width = 0.35

fig, ax = plt.subplots(figsize=(10,6))
ax.bar(x - width/2, actual_counts, width, label='Actual', color='skyblue')
ax.bar(x + width/2, pred_counts, width, label='Predicted', color='lightcoral')

ax.set_xlabel('Sentiment')
ax.set_ylabel('Count')
ax.set_title('Predicted vs Actual Sentiment Distribution')
ax.set_xticks(x)
ax.set_xticklabels(sentiments)
ax.legend()

plt.tight_layout()
plt.show()